# Constants, packages and path

In [59]:
# constants for LPG supply chain processing
default_LPG_price = 0.6  # USD/kg
STS_cost = 0.1  # 10%
pre_bottling_cost = 0.2  # 20% TO BE MODIFIED
CRUDE_BARREL_TO_TONS = 7.33  # barrel of crude per ton
BUFFER_KM = 10  # km radius for spatial buffer queries

import pandas as pd
import geopandas as gpd
from pathlib import Path
import numpy as np
import rasterio
from rasterio.mask import mask as raster_mask
from shapely.geometry import Point, box

# Set up data path
data_dir = Path("dataset_first_step")
data_dir.mkdir(exist_ok=True)

# create mask layer to use as base (just to try)

In [61]:
# Build mask_layer from friction raster footprint
from rasterio.features import shapes
from shapely.geometry import shape
from shapely.ops import unary_union

friction_mask_path = data_dir / "friction_moto.tif"
if not friction_mask_path.exists():
    raise FileNotFoundError(f"Missing friction raster for mask creation: {friction_mask_path}")

with rasterio.open(friction_mask_path) as src:
    if src.crs is None:
        raise ValueError("friction_moto.tif has no CRS")

    # Valid data footprint: non-nodata pixels
    valid_mask = src.dataset_mask() > 0
    if not np.any(valid_mask):
        raise ValueError("friction_moto.tif contains no valid pixels to build mask_layer")

    mask_shapes = [
        shape(geom)
        for geom, value in shapes(valid_mask.astype(np.uint8), mask=valid_mask, transform=src.transform)
        if value == 1
    ]

    if len(mask_shapes) == 0:
        raise ValueError("Could not extract a valid geometry footprint from friction_moto.tif")

    mask_geom = unary_union(mask_shapes)
    mask_layer = gpd.GeoDataFrame(geometry=[mask_geom], crs=src.crs).to_crs("EPSG:4326")

print(f"Mask layer initialized from friction raster ({len(mask_layer)} geometry)")
print("Constants declared and imports completed")

Mask layer initialized from friction raster (1 geometry)
Constants declared and imports completed


# Create useful function for data loading and fill missing values

In [65]:
# Load input data files with smart defaults
from onstove.layer import VectorLayer, RasterLayer
from shapely.ops import nearest_points

# Define default gpkg and layer names
DEFAULT_GPKG = data_dir / "default.gpkg"
DEFAULT_LAYERS = {
    'refineries': 'refineries',
    'ports': 'ports',
    'gas_plants': 'gas_plants',
    'primary_storage': 'primary_storage',
    'border_points': 'border_points',
}


def _as_vector_layer(gdf, name="mask"):
    vect = VectorLayer(name=name)
    vect.data = gdf.copy()
    return vect


def load_raster_means(raster_path, band_names, mask_gdf):
    """
    Read a multiband raster, clip by mask_gdf, and return band averages.

    Output format matches previous table usage:
    - index: band names
    - single numeric column
    """
    if not raster_path.exists():
        raise FileNotFoundError(f"Missing raster file: {raster_path}")

    if mask_gdf.crs is None:
        raise ValueError("mask_layer has no CRS")

    mask_vect = _as_vector_layer(mask_gdf, name="mask_layer")
    values = {}

    for band_idx, band_name in enumerate(band_names, start=1):
        raster = RasterLayer(name=f"{raster_path.stem}_{band_name}", path=str(raster_path), band=band_idx)
        raster.mask(mask_vect, crop=True, all_touched=False)

        arr = raster.data.astype(np.float64)
        nodata = raster.meta.get('nodata')
        if nodata is not None:
            try:
                if not np.isnan(nodata):
                    arr[arr == nodata] = np.nan
            except TypeError:
                arr[arr == nodata] = np.nan

        mean_val = np.nanmean(arr)
        if np.isnan(mean_val):
            raise ValueError(
                f"Band '{band_name}' in {raster_path.name} has no valid data inside mask_layer"
            )
        values[band_name] = float(mean_val)

    result = pd.DataFrame.from_dict(values, orient='index')
    result.columns = [1]
    return result


def load_or_use_default(gpkg_name, default_gpkg_path, default_layer_name, mask_gdf, data_dir=data_dir):
    """
    SMART DATA LOADING with fallback to defaults

    Two-phase strategy:
    1. If user provided data (file exists in dataset_first_step/), load it directly
    2. If not, load default layer from default.gpkg, clip it with mask_gdf, and keep it in memory
    """
    gpkg_path = data_dir / f"{gpkg_name}.gpkg"
    mask_vect = _as_vector_layer(mask_gdf, name="mask_layer")

    if gpkg_path.exists():
        # PHASE 1: User data found
        layer = VectorLayer(name=gpkg_name, path=str(gpkg_path))
        layer.mask(mask_vect, keep_geom_type=False)
        gdf = layer.data
        print(f"{gpkg_name}: loaded and clipped in memory ({len(gdf)} records)")
        return gdf

    # PHASE 2: Use default.gpkg layer and clip with mask_layer (no file creation)
    if not default_gpkg_path.exists():
        raise FileNotFoundError(f"Missing default gpkg: {default_gpkg_path}")

    default_gdf = gpd.read_file(default_gpkg_path, layer=default_layer_name)
    default_layer = _as_vector_layer(default_gdf, name=gpkg_name)
    default_layer.mask(mask_vect, keep_geom_type=False)

    gdf = default_layer.data
    print(f"{gpkg_name}: clipped from default.gpkg/{default_layer_name} in memory ({len(gdf)} records)")
    return gdf


def load_border_points_or_default(mask_gdf, data_dir=data_dir, near_km=5):
    """
    Load border_points from user file if provided, otherwise from default.gpkg.

    Spatial rule:
    - Keep points inside mask_layer as-is.
    - If outside but within near_km from mask_layer, snap to nearest point on mask boundary.
    - Drop points farther than near_km from mask_layer.
    """
    user_path = data_dir / "border_points.gpkg"

    if user_path.exists():
        try:
            points = gpd.read_file(user_path, layer="border_points")
        except Exception:
            points = gpd.read_file(user_path)
        source_name = f"{user_path.name}"
    else:
        if not DEFAULT_GPKG.exists():
            raise FileNotFoundError(f"Missing default gpkg: {DEFAULT_GPKG}")
        points = gpd.read_file(DEFAULT_GPKG, layer=DEFAULT_LAYERS['border_points'])
        source_name = f"default.gpkg/{DEFAULT_LAYERS['border_points']}"

    if points.empty:
        print(f"border_points: loaded 0 records from {source_name}")
        return points

    if mask_gdf.crs is None:
        raise ValueError("mask_layer has no CRS")
    if points.crs is None:
        raise ValueError("border_points has no CRS")

    pts = points.to_crs(mask_gdf.crs).copy()
    mask_union = mask_gdf.geometry.unary_union

    pts_m = pts.to_crs("EPSG:3857").copy()
    mask_m = mask_gdf.to_crs("EPSG:3857")
    mask_union_m = mask_m.geometry.unary_union

    inside_count = 0
    moved_count = 0
    dropped_count = 0

    keep_rows = []
    new_geom_m = []

    for idx, geom_m in zip(pts_m.index, pts_m.geometry):
        if geom_m is None or geom_m.is_empty:
            dropped_count += 1
            continue

        if mask_union_m.covers(geom_m):
            inside_count += 1
            keep_rows.append(idx)
            new_geom_m.append(geom_m)
            continue

        dist_m = float(geom_m.distance(mask_union_m))
        if dist_m <= near_km * 1000:
            snapped = nearest_points(geom_m, mask_union_m)[1]
            moved_count += 1
            keep_rows.append(idx)
            new_geom_m.append(snapped)
        else:
            dropped_count += 1

    if len(keep_rows) == 0:
        out = pts.iloc[0:0].copy()
    else:
        out_m = pts_m.loc[keep_rows].copy()
        out_m.geometry = new_geom_m
        out = out_m.to_crs(mask_gdf.crs)

    print(
        f"border_points: loaded from {source_name} | inside={inside_count}, moved_to_mask={moved_count}, dropped_far={dropped_count}"
    )
    return out


def fill_from_buffer(gdf, default_gdf, cols, buffer_km=BUFFER_KM):
    """
    FILL MISSING VALUES using spatial buffer queries

    For each geometry in gdf with NaN values in 'cols', search for the nearest value
    in default_gdf within a radius of buffer_km.
    """
    gdf, default_gdf = gdf.copy(), default_gdf.to_crs(gdf.crs)
    buffer_dist = buffer_km / 111.0 if gdf.crs.is_geographic else buffer_km * 1000

    for col in cols:
        if col not in gdf.columns:
            continue
        missing_idx = gdf[gdf[col].isna() | (gdf[col] == '')].index
        if len(missing_idx) == 0:
            continue
        print(f"  Filling '{col}': {len(missing_idx)} rows")

        for idx in missing_idx:
            port_buffer = gdf.loc[idx, 'geometry'].buffer(buffer_dist)
            nearby = default_gdf[default_gdf.geometry.intersects(port_buffer)]

            if len(nearby) > 0 and col in nearby.columns:
                val = nearby[col].dropna()
                if len(val) > 0:
                    gdf.at[idx, col] = val.iloc[0]
    return gdf

# redistribution of small national shares

In [71]:
def redistribute_small_national_shares(nat_shares_df, min_share_pct=1.0):
    """
    Redistribute national shares when a supply mode is below a minimum percentage.

    Rules:
    - Modes with share < min_share_pct are set to 0.
    - Their share is redistributed proportionally across remaining modes.
    - Final shares sum to 100.
    """
    if nat_shares_df.shape[1] == 0:
        raise ValueError("nat_shares_df has no numeric column")

    out = nat_shares_df.copy()
    share_col = out.columns[0]

    shares = pd.to_numeric(out[share_col], errors='coerce').fillna(0.0)
    keep = shares >= float(min_share_pct)

    if keep.sum() == 0:
        raise ValueError(
            f"All supply modes are below the threshold ({min_share_pct}%). Cannot redistribute."
        )

    redistributed = shares.where(keep, 0.0)
    total_keep = float(redistributed.sum())

    if total_keep <= 0:
        raise ValueError("No positive share available after threshold filtering.")

    redistributed = redistributed * (100.0 / total_keep)
    out[share_col] = redistributed
    return out


# data loading or default data use

In [72]:
# Load all data
try:
    refineries = load_or_use_default("refineries", DEFAULT_GPKG, DEFAULT_LAYERS['refineries'], mask_layer)
    ports = load_or_use_default("ports", DEFAULT_GPKG, DEFAULT_LAYERS['ports'], mask_layer)
    gas_plants = load_or_use_default("gas_plants", DEFAULT_GPKG, DEFAULT_LAYERS['gas_plants'], mask_layer)
    primary_storage = load_or_use_default("primary_storage", DEFAULT_GPKG, DEFAULT_LAYERS['primary_storage'], mask_layer)
    border_points = load_border_points_or_default(mask_layer)

    # Read multiband rasters and use average over mask_layer as actual values
    national_shares = load_raster_means(
        data_dir / "national_shares.tif",
        ['ports', 'refineries', 'gas_plants', 'border_points'],
        mask_layer,
    )
    lpg_import_cost = load_raster_means(
        data_dir / "lpg_import_cost.tif",
        ['import_land', 'import_sea'],
        mask_layer,
    )

    print("\nNational shares (mean over masked area, raw):")
    print(national_shares)

    # Redistribute shares below 1% across the remaining supply modes
    national_shares = redistribute_small_national_shares(national_shares, min_share_pct=1.0)

    print("\nNational shares (after redistribution of modes < 1%):")
    print(national_shares)

    print("\nLPG import costs (mean over masked area):")
    print(lpg_import_cost)

    # Load defaults once (reuse for all fillings)
    default_data = {name: gpd.read_file(DEFAULT_GPKG, layer=layer_name) for name, layer_name in DEFAULT_LAYERS.items()}

    print("\n✓ Data loaded successfully")
except (FileNotFoundError, ValueError) as e:
    print(f"Data loading error: {e}")

refineries: clipped from default.gpkg/refineries in memory (4 records)
ports: clipped from default.gpkg/ports in memory (11 records)
gas_plants: clipped from default.gpkg/gas_plants in memory (5 records)
primary_storage: clipped from default.gpkg/primary_storage in memory (19 records)


C:\Users\Fra\AppData\Local\Temp\ipykernel_14736\3732348689.py:129: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
C:\Users\Fra\AppData\Local\Temp\ipykernel_14736\3732348689.py:133: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
d:\Anaconda\envs\onstove2\Lib\site-packages\onstove\layer.py:946: UserWarning: The national_shares_ports layer do not have a defined nodata value, thus np.nan was assigned. You can change this defining the nodata value in the metadata of the variable as: variable.meta['nodata'] = value
d:\Anaconda\envs\onstove2\Lib\site-packages\onstove\layer.py:946: UserWarning: The national_shares_refineries layer do not have a defined nodata value, thus np.nan was assigned. You can change this defining the nodata value in the metadata of the variable as: variable.meta['nodata'] = value


border_points: loaded from default.gpkg/border_points | inside=3, moved_to_mask=3, dropped_far=176

National shares (mean over masked area, raw):
                       1
ports          50.716459
refineries      0.063389
gas_plants     49.198556
border_points   0.021596

National shares (after redistribution of modes < 1%):
                       1
ports          50.759597
refineries      0.000000
gas_plants     49.240403
border_points   0.000000

LPG import costs (mean over masked area):
                    1
import_land  0.094122
import_sea   0.019422

✓ Data loaded successfully


d:\Anaconda\envs\onstove2\Lib\site-packages\onstove\layer.py:946: UserWarning: The national_shares_gas_plants layer do not have a defined nodata value, thus np.nan was assigned. You can change this defining the nodata value in the metadata of the variable as: variable.meta['nodata'] = value
d:\Anaconda\envs\onstove2\Lib\site-packages\onstove\layer.py:946: UserWarning: The national_shares_border_points layer do not have a defined nodata value, thus np.nan was assigned. You can change this defining the nodata value in the metadata of the variable as: variable.meta['nodata'] = value
d:\Anaconda\envs\onstove2\Lib\site-packages\onstove\layer.py:946: UserWarning: The lpg_import_cost_import_land layer do not have a defined nodata value, thus np.nan was assigned. You can change this defining the nodata value in the metadata of the variable as: variable.meta['nodata'] = value
d:\Anaconda\envs\onstove2\Lib\site-packages\onstove\layer.py:946: UserWarning: The lpg_import_cost_import_sea layer do n

# fill missing data from buffer or default + calculate port cost

In [77]:
# Process refineries: fill missing values and ensure key fields

def fill_lpg_price_from_raster(gdf, raster_path, band_index=1, overwrite=False):
    """
    Fill LPG_price by sampling a raster at each facility location.

    - Uses geometry centroid for non-point features.
    - If overwrite=False, only fills missing LPG_price values.
    """
    out = gdf.copy()

    if out.crs is None:
        raise ValueError("Input GeoDataFrame has no CRS")

    if 'LPG_price' not in out.columns:
        out['LPG_price'] = np.nan

    raster_file = Path(raster_path)
    if not raster_file.exists():
        raise FileNotFoundError(f"Missing LPG price raster: {raster_file}")

    with rasterio.open(raster_file) as src:
        if src.crs is None:
            raise ValueError("LPG price raster has no CRS")
        if band_index < 1 or band_index > src.count:
            raise ValueError(
                f"Requested band {band_index} for {raster_file.name}, but raster has {src.count} band(s)"
            )

        work = out.to_crs(src.crs).copy()
        geom_series = work.geometry
        if not work.geom_type.eq('Point').all():
            geom_series = work.geometry.centroid

        sampled_values = np.full(len(work), np.nan, dtype=np.float64)
        valid_positions = []
        valid_coords = []

        for pos, geom in enumerate(geom_series):
            if geom is None or geom.is_empty:
                continue
            valid_positions.append(pos)
            valid_coords.append((geom.x, geom.y))

        if len(valid_coords) > 0:
            for pos, sample in zip(valid_positions, src.sample(valid_coords, indexes=band_index)):
                sampled_values[pos] = float(sample[0])

        nodata = src.nodata
        if nodata is not None:
            sampled_values[sampled_values == nodata] = np.nan

    sampled_series = pd.Series(sampled_values, index=out.index)
    if overwrite:
        mask = sampled_series.notna()
    else:
        mask = out['LPG_price'].isna() & sampled_series.notna()

    out.loc[mask, 'LPG_price'] = sampled_series.loc[mask]
    return out


def _as_bool_series(series):
    def _to_bool(v):
        if isinstance(v, bool):
            return v
        if pd.isna(v):
            return False
        s = str(v).strip().lower()
        return s in {'true', '1', 'yes', 'y', 't'}

    return series.apply(_to_bool)


def process_refineries(gdf, default_gdf, lpg_price_raster_path, price_band=1):
    """
    REFINERIES PROCESSING

    Fill missing values from buffer and ensure key fields.

    LPG_price fallback is sampled from fob_per_kg raster at facility location.
    """
    gdf = gdf.copy()

    # Phase 1: Fill from buffer
    fill_cols = ['LPG_prod', 'LPG_price']
    gdf = fill_from_buffer(gdf, default_gdf, fill_cols)

    # Phase 2: Ensure production and fill LPG_price from raster
    if 'LPG_prod' not in gdf.columns:
        gdf['LPG_prod'] = 0.0
    else:
        gdf['LPG_prod'] = gdf['LPG_prod'].fillna(0.0)

    gdf = fill_lpg_price_from_raster(gdf, lpg_price_raster_path, band_index=price_band, overwrite=False)

    missing_prices = int(gdf['LPG_price'].isna().sum())
    if missing_prices > 0:
        print(f"Warning: {missing_prices} refinery rows still have missing LPG_price after raster sampling")

    return gdf


def process_ports(gdf, default_gdf, storage_gdf, import_cost_df, lpg_price_raster_path, price_band=1):
    """
    PORTS PROCESSING

    Fill missing values, find nearest storage facility, and adjust pricing based on compliance.

    Missing LPG_price values are filled from fob_per_kg raster at each port location.
    """
    gdf = gdf.copy()

    # Phase 1: Fill missing values from buffer
    fill_cols = ['VLGC_compliance', 'LPG_price']
    gdf = fill_from_buffer(gdf, default_gdf, fill_cols)

    # Phase 2: Ensure LPG_price exists and fill from raster values
    if 'LPG_price' not in gdf.columns:
        gdf['LPG_price'] = np.nan

    gdf = fill_lpg_price_from_raster(gdf, lpg_price_raster_path, band_index=price_band, overwrite=False)

    # Phase 3: Initialize port-specific fields
    gdf['LPG_capacity'] = 0.0
    gdf['tanks_nearby'] = 0.0
    gdf['LPG_compliance'] = False

    # Ensure VLGC_compliance exists and fill NaN with False
    if 'VLGC_compliance' not in gdf.columns:
        gdf['VLGC_compliance'] = False
    else:
        gdf['VLGC_compliance'] = gdf['VLGC_compliance'].fillna(False)

    gdf = gdf.to_crs(storage_gdf.crs) if storage_gdf is not None else gdf

    # Phase 4: Find nearby storage facilities and compute de-duplicated LPG_capacity
    if storage_gdf is not None:
        buffer_dist = BUFFER_KM / 111.0 if gdf.crs.is_geographic else BUFFER_KM * 1000

        # Precompute buffers for all ports
        port_buffers = {idx: geom.buffer(buffer_dist) for idx, geom in gdf.geometry.items()}

        # Count how many ports are connected to each storage (within buffer)
        storage_connection_counts = {}
        for s_idx, s_row in storage_gdf.iterrows():
            s_geom = s_row.geometry
            n_connected_ports = sum(1 for p_buf in port_buffers.values() if p_buf.intersects(s_geom))
            storage_connection_counts[s_idx] = n_connected_ports

        # For each port, sum nearby capacities after dividing each storage by its number of connected ports
        for idx, port_buffer in port_buffers.items():
            nearby = storage_gdf[storage_gdf.geometry.intersects(port_buffer)]

            if len(nearby) > 0:
                allocated_caps = []
                for s_idx, s_row in nearby.iterrows():
                    raw_cap = float(s_row['LPG_capacity']) if ('LPG_capacity' in nearby.columns and pd.notna(s_row.get('LPG_capacity'))) else 0.0
                    n_conn = storage_connection_counts.get(s_idx, 0)
                    if n_conn > 0:
                        allocated_caps.append(raw_cap / n_conn)

                gdf.at[idx, 'LPG_capacity'] = float(sum(allocated_caps))
                gdf.at[idx, 'tanks_nearby'] = len(nearby)
                gdf.at[idx, 'LPG_compliance'] = True
            else:
                gdf.at[idx, 'LPG_capacity'] = 0.0
                gdf.at[idx, 'tanks_nearby'] = 0
                gdf.at[idx, 'LPG_compliance'] = False

    # Phase 5: Price adjustment based on LPG_compliance
    import_sea = import_cost_df.loc['import_sea'].iloc[0]

    # Ports WITH storage nearby (LPG_compliance = True)
    compliant = gdf['LPG_compliance'] == True
    gdf.loc[compliant, 'LPG_price'] = gdf.loc[compliant, 'LPG_price'] * (1 + import_sea)
    # Add STS_cost for non-VLGC compliant ports
    non_vlgc_compliant = compliant & (gdf['VLGC_compliance'] != True)
    gdf.loc[non_vlgc_compliant, 'LPG_price'] *= (1 + STS_cost)

    # Ports WITHOUT storage nearby (LPG_compliance = False)
    non_compliant = gdf['LPG_compliance'] == False
    gdf.loc[non_compliant, 'LPG_price'] = gdf.loc[non_compliant, 'LPG_price'] * (1 + import_sea) * (1 + pre_bottling_cost)

    missing_prices = int(gdf['LPG_price'].isna().sum())
    if missing_prices > 0:
        print(f"Warning: {missing_prices} port rows still have missing LPG_price after raster sampling")

    return gdf


def process_primary_storage(gdf, default_gdf, lpg_price_raster_path, price_band=1):
    """
    PRIMARY STORAGE PROCESSING

    Fill missing values from buffer and ensure key fields.

    LPG_price fallback is sampled from fob_per_kg raster at facility location.
    """
    gdf = gdf.copy()

    # Phase 1: Fill from buffer
    fill_cols = ['LPG_capacity', 'LPG_price']
    gdf = fill_from_buffer(gdf, default_gdf, fill_cols)

    # Phase 2: Ensure storage capacity and fill LPG_price from raster
    if 'LPG_capacity' not in gdf.columns:
        gdf['LPG_capacity'] = 0.0
    else:
        gdf['LPG_capacity'] = gdf['LPG_capacity'].fillna(0.0)

    gdf = fill_lpg_price_from_raster(gdf, lpg_price_raster_path, band_index=price_band, overwrite=False)

    missing_prices = int(gdf['LPG_price'].isna().sum())
    if missing_prices > 0:
        print(f"Warning: {missing_prices} primary storage rows still have missing LPG_price after raster sampling")

    return gdf


def process_border_points(gdf, nat_shares_df, lpg_price_raster_path, price_band=1):
    """
    BORDER POINTS PROCESSING

    - Ensure LPG_price exists (sample from fob_per_kg if missing)
    - Compute percentage so that sum equals national border_points share
    - If LPG_capacity exists and has positive total, use it for proportions
    - Otherwise use equal share, but osbp=true gets double weight
    """
    bp = gdf.copy()

    if bp.empty:
        bp['LPG_price'] = pd.Series(dtype='float64')
        bp['percentage'] = pd.Series(dtype='float64')
        bp['category'] = pd.Series(dtype='object')
        return bp

    if 'LPG_price' not in bp.columns:
        bp['LPG_price'] = np.nan

    bp = fill_lpg_price_from_raster(bp, lpg_price_raster_path, band_index=price_band, overwrite=False)

    border_share = float(nat_shares_df.loc['border_points'].iloc[0]) / 100.0

    if 'LPG_capacity' in bp.columns:
        cap = pd.to_numeric(bp['LPG_capacity'], errors='coerce').fillna(0.0)
        if cap.sum() > 0:
            weights = cap
        else:
            if 'osbp' in bp.columns:
                osbp_flag = _as_bool_series(bp['osbp'])
                weights = pd.Series(np.where(osbp_flag, 2.0, 1.0), index=bp.index, dtype='float64')
            else:
                weights = pd.Series(1.0, index=bp.index, dtype='float64')
    else:
        if 'osbp' in bp.columns:
            osbp_flag = _as_bool_series(bp['osbp'])
            weights = pd.Series(np.where(osbp_flag, 2.0, 1.0), index=bp.index, dtype='float64')
        else:
            weights = pd.Series(1.0, index=bp.index, dtype='float64')

    weight_sum = float(weights.sum())
    if weight_sum > 0:
        bp['percentage'] = border_share * (weights / weight_sum)
    else:
        bp['percentage'] = 0.0

    bp['category'] = 'border_points'

    if 'country' not in bp.columns:
        bp['country'] = 'Unknown'
    else:
        bp['country'] = bp['country'].fillna('Unknown')

    missing_prices = int(bp['LPG_price'].isna().sum())
    if missing_prices > 0:
        print(f"Warning: {missing_prices} border_points rows still have missing LPG_price after raster sampling")

    print(f"border_points: total percentage assigned = {bp['percentage'].sum():.2%}")
    return bp


def assign_nearest_tank_by_traveltime(facilities_gdf, storage_gdf, friction_path):
    """
    Assign nearest primary storage by travel time using OnStove friction routing.

    Adds three fields:
    - tank_name: nearest storage name by minimum travel time
    - tank_distance: straight-line distance in meters to selected storage
    - tank_traveltime: modeled travel time in hours to selected storage
    """
    facilities = facilities_gdf.copy()

    if facilities.empty:
        facilities['tank_name'] = pd.Series(dtype='object')
        facilities['tank_distance'] = pd.Series(dtype='float64')
        facilities['tank_traveltime'] = pd.Series(dtype='float64')
        return facilities

    if storage_gdf is None or storage_gdf.empty:
        facilities['tank_name'] = None
        facilities['tank_distance'] = np.nan
        facilities['tank_traveltime'] = np.nan
        return facilities

    friction_file = Path(friction_path)
    if not friction_file.exists():
        raise FileNotFoundError(f"Missing friction raster: {friction_file}")

    if facilities.crs is None:
        raise ValueError("Facilities layer has no CRS")
    if storage_gdf.crs is None:
        raise ValueError("primary_storage layer has no CRS")

    friction = RasterLayer(name='friction_surface', path=str(friction_file))
    friction_crs = friction.meta.get('crs')
    if friction_crs is None:
        raise ValueError("Friction raster has no CRS in metadata")

    facilities_tt = facilities.to_crs(friction_crs).copy()
    storage_tt = storage_gdf.to_crs(friction_crs).copy().reset_index(drop=True)

    name_col = 'name' if 'name' in storage_tt.columns else None
    if name_col is None:
        for col in storage_tt.columns:
            if str(col).lower() == 'name':
                name_col = col
                break
    if name_col is None:
        storage_tt['__tank_name'] = [f'primary_storage_{i}' for i in storage_tt.index]
        name_col = '__tank_name'
    else:
        storage_tt[name_col] = storage_tt[name_col].astype(str)
        missing_names = storage_tt[name_col].str.strip().isin(['', 'nan', 'None'])
        storage_tt.loc[missing_names, name_col] = [f'primary_storage_{i}' for i in storage_tt.index[missing_names]]

    facility_points = facilities_tt.geometry
    if not facilities_tt.geom_type.eq('Point').all():
        facility_points = facilities_tt.geometry.centroid

    storage_points = storage_tt.geometry
    if not storage_tt.geom_type.eq('Point').all():
        storage_points = storage_tt.geometry.centroid

    facility_rows = []
    facility_cols = []
    for geom in facility_points:
        row, col = rasterio.transform.rowcol(friction.meta['transform'], geom.x, geom.y)
        facility_rows.append(row)
        facility_cols.append(col)

    travel_matrix = np.full((len(facilities_tt), len(storage_tt)), np.nan, dtype=np.float64)

    for tank_idx, tank_geom in enumerate(storage_points):
        tank_layer = VectorLayer(name=f"tank_{tank_idx}")
        tank_layer.data = gpd.GeoDataFrame({'geometry': [tank_geom]}, geometry='geometry', crs=storage_tt.crs)

        tt_raster = tank_layer.travel_time(friction=friction, create_raster=False)
        tt_data = tt_raster.data
        max_row, max_col = tt_data.shape

        for fac_idx, (row, col) in enumerate(zip(facility_rows, facility_cols)):
            if 0 <= row < max_row and 0 <= col < max_col:
                val = tt_data[row, col]
                if np.isfinite(val):
                    travel_matrix[fac_idx, tank_idx] = float(val)

    tank_name = np.array([None] * len(facilities_tt), dtype=object)
    tank_distance = np.full(len(facilities_tt), np.nan, dtype=np.float64)
    tank_traveltime = np.full(len(facilities_tt), np.nan, dtype=np.float64)

    valid = np.any(np.isfinite(travel_matrix), axis=1)
    if np.any(valid):
        best_tank_idx = np.nanargmin(travel_matrix[valid], axis=1)
        valid_rows = np.where(valid)[0]

        storage_names = storage_tt[name_col].astype(str).to_numpy()
        tank_name[valid] = storage_names[best_tank_idx]
        tank_traveltime[valid] = travel_matrix[valid_rows, best_tank_idx]

        facilities_m = facilities_tt.to_crs('EPSG:3857')
        storage_m = storage_tt.to_crs('EPSG:3857')
        facility_points_m = facilities_m.geometry if facilities_m.geom_type.eq('Point').all() else facilities_m.geometry.centroid
        storage_points_m = storage_m.geometry if storage_m.geom_type.eq('Point').all() else storage_m.geometry.centroid

        for row_idx, tank_idx in zip(valid_rows, best_tank_idx):
            tank_distance[row_idx] = float(facility_points_m.iloc[row_idx].distance(storage_points_m.iloc[tank_idx]))

    facilities['tank_name'] = tank_name
    facilities['tank_distance'] = tank_distance
    facilities['tank_traveltime'] = tank_traveltime
    return facilities

# Apply processing for each facility type
lpg_price_raster = data_dir / 'fob_per_kg.tif'
refineries = process_refineries(refineries, default_data['refineries'], lpg_price_raster_path=lpg_price_raster, price_band=1)
primary_storage = process_primary_storage(primary_storage, default_data['primary_storage'], lpg_price_raster_path=lpg_price_raster, price_band=1)
ports = process_ports(ports, default_data['ports'], primary_storage, lpg_import_cost, lpg_price_raster_path=lpg_price_raster, price_band=1)
gas_plants = process_refineries(gas_plants, default_data['gas_plants'], lpg_price_raster_path=lpg_price_raster, price_band=1)
border_points = process_border_points(border_points, national_shares, lpg_price_raster_path=lpg_price_raster, price_band=1)

# Assign nearest primary storage by travel time (OnStove)
friction_path = data_dir / 'friction_moto.tif'
refineries = assign_nearest_tank_by_traveltime(refineries, primary_storage, friction_path)
ports = assign_nearest_tank_by_traveltime(ports, primary_storage, friction_path)
gas_plants = assign_nearest_tank_by_traveltime(gas_plants, primary_storage, friction_path)
border_points = assign_nearest_tank_by_traveltime(border_points, primary_storage, friction_path)

print("✓ Facilities processed in memory")
print("✓ LPG_price filled from fob_per_kg.tif at facility locations")
print("✓ border_points processed (snapping + percentage + LPG_price)")
print("✓ tank_name, tank_distance and tank_traveltime added to refinery/port/gas/border layers")

border_points: total percentage assigned = 0.00%
✓ Facilities processed in memory
✓ LPG_price filled from fob_per_kg.tif at facility locations
✓ border_points processed (snapping + percentage + LPG_price)
✓ tank_name, tank_distance and tank_traveltime added to refinery/port/gas/border layers


# calculate market share %

In [74]:
# STEP 4: Calculate percentages
def calculate_percentages(refineries_gdf, ports_gdf, gas_plants_gdf, border_points_gdf, nat_shares_df, storage_gdf):
    """
    CALCULATE FACILITY-LEVEL MARKET SHARE PERCENTAGES

    Computes market share by distributing national shares across individual facilities
    based on their production/storage capacity.

    border_points_gdf is expected to already contain a valid 'percentage' column
    whose sum matches the national border_points share.

    OUTPUT:
    - result: combined GeoDataFrame for quick checks/summary
    - by_category: dict with one GeoDataFrame per source category, preserving original fields
    """

    # Choose a common CRS for all facility layers before concatenation
    target_crs = refineries_gdf.crs or ports_gdf.crs or gas_plants_gdf.crs or border_points_gdf.crs

    # For refineries, use LPG_capacity for proportional split; if missing, fallback to crude_capacity
    refineries_weight_col = 'LPG_capacity' if 'LPG_capacity' in refineries_gdf.columns else 'crude_capacity'

    def _numeric_series_or_zero(gdf, col):
        if col in gdf.columns:
            return pd.to_numeric(gdf[col], errors='coerce').fillna(0.0)
        return pd.Series(0.0, index=gdf.index, dtype='float64')

    # Calculate totals
    totals = {
        'refineries': _numeric_series_or_zero(refineries_gdf, refineries_weight_col).sum(),
        'ports': _numeric_series_or_zero(ports_gdf, 'LPG_capacity').sum(),
        'gas_plants': _numeric_series_or_zero(gas_plants_gdf, 'LPG_prod').sum(),
    }

    # Get national shares from raster-derived values (no fallback defaults)
    nat_shares = {
        'refineries': nat_shares_df.loc['refineries'].iloc[0] / 100,
        'ports': nat_shares_df.loc['ports'].iloc[0] / 100,
        'gas_plants': nat_shares_df.loc['gas_plants'].iloc[0] / 100,
        'border_points': nat_shares_df.loc['border_points'].iloc[0] / 100,
    }

    print(f"Refineries proportional column: {refineries_weight_col}")
    print(f"Totals: Refineries={totals['refineries']:.0f} | Ports={totals['ports']:.0f} | Gas={totals['gas_plants']:.0f}")
    print(f"Shares: {nat_shares}")

    by_category = {}
    specs = [
        ('refineries', refineries_gdf, refineries_weight_col),
        ('ports', ports_gdf, 'LPG_capacity'),
        ('gas_plants', gas_plants_gdf, 'LPG_prod'),
    ]

    for src_name, src_gdf, weight_col in specs:
        gdf = src_gdf.copy()

        # Reproject each layer to a common CRS to avoid concat CRS conflicts
        if target_crs is not None and gdf.crs is not None and gdf.crs != target_crs:
            gdf = gdf.to_crs(target_crs)

        # Ensure required columns used later in the pipeline are present
        if 'name' not in gdf.columns:
            gdf['name'] = [f'{src_name}_{idx}' for idx in gdf.index]

        if 'LPG_price' not in gdf.columns:
            gdf['LPG_price'] = np.nan
        else:
            gdf['LPG_price'] = pd.to_numeric(gdf['LPG_price'], errors='coerce')

        if 'country' not in gdf.columns:
            gdf['country'] = 'Unknown'
        else:
            gdf['country'] = gdf['country'].fillna('Unknown')

        # Keep category in each layer and compute percentage
        gdf['category'] = src_name if src_name != 'gas_plants' else 'gasplants'

        weights = _numeric_series_or_zero(gdf, weight_col)

        if totals[src_name] > 0:
            gdf['percentage'] = nat_shares[src_name] * (weights / totals[src_name])
        else:
            gdf['percentage'] = 0.0

        by_category[src_name] = gdf

    # Border points: keep precomputed percentages from processing step
    border = border_points_gdf.copy()
    if target_crs is not None and border.crs is not None and border.crs != target_crs:
        border = border.to_crs(target_crs)

    if 'name' not in border.columns:
        border['name'] = [f'border_points_{idx}' for idx in border.index]

    if 'LPG_price' not in border.columns:
        border['LPG_price'] = np.nan
    else:
        border['LPG_price'] = pd.to_numeric(border['LPG_price'], errors='coerce')

    if 'country' not in border.columns:
        border['country'] = 'Unknown'
    else:
        border['country'] = border['country'].fillna('Unknown')

    border['category'] = 'border_points'
    if 'percentage' not in border.columns:
        border['percentage'] = 0.0
    else:
        border['percentage'] = pd.to_numeric(border['percentage'], errors='coerce').fillna(0.0)

    by_category['border_points'] = border

    result = gpd.GeoDataFrame(
        pd.concat(
            [
                by_category['refineries'],
                by_category['ports'],
                by_category['gas_plants'],
                by_category['border_points'],
            ],
            ignore_index=True,
        ),
        crs=target_crs,
    )

    print(f"border_points share in result: {by_category['border_points']['percentage'].sum():.2%}")
    print(f"✓ {len(result)} sources calculated | Total share: {result['percentage'].sum():.2%}")
    return result, by_category

# Use in-memory processed data
beginning, facilities_by_category = calculate_percentages(refineries, ports, gas_plants, border_points, national_shares, primary_storage)


Refineries proportional column: crude_capacity
Totals: Refineries=446 | Ports=686125 | Gas=1445000
Shares: {'refineries': np.float64(0.0), 'ports': np.float64(0.507595971268243), 'gas_plants': np.float64(0.4924040287317571), 'border_points': np.float64(0.0)}
border_points share in result: 0.00%
✓ 26 sources calculated | Total share: 100.00%


# save output

In [75]:
# STEP 5: Save output as a multi-layer GPKG (facility layers + primary storage + border points)
output_gpkg = data_dir / "beginning.gpkg"
output_gpkg_fallback = data_dir / "beginning_border_points.gpkg"

layer_map = [
    ('refineries', 'refineries'),
    ('ports', 'ports'),
    ('gas_plants', 'gas_plants'),
    ('primary_storage', 'primary_storage'),
    ('border_points', 'border_points'),
]

export_layers = {
    'refineries': facilities_by_category['refineries'],
    'ports': facilities_by_category['ports'],
    'gas_plants': facilities_by_category['gas_plants'],
    'primary_storage': primary_storage,
    'border_points': facilities_by_category['border_points'],
}

def _sanitize_for_gpkg(gdf):
    """Make a GeoDataFrame safe for GPKG writing while preserving all fields."""
    cleaned = gdf.copy()

    # Avoid case-insensitive duplicate field names in GPKG
    seen = {}
    new_cols = []
    for col in cleaned.columns:
        key = str(col).lower()
        if key in seen:
            seen[key] += 1
            new_cols.append(f"{col}_{seen[key]}")
        else:
            seen[key] = 0
            new_cols.append(col)
    cleaned.columns = new_cols

    # Convert problematic object fields to strings to avoid writer errors
    geom_col = cleaned.geometry.name if hasattr(cleaned, 'geometry') else 'geometry'
    for col in cleaned.columns:
        if col == geom_col:
            continue
        if cleaned[col].dtype == 'object':
            cleaned[col] = cleaned[col].apply(
                lambda v: v if isinstance(v, (str, int, float, bool, type(None), np.integer, np.floating)) else str(v)
            )

    return cleaned


def _write_layers(target_path):
    if target_path.exists():
        target_path.unlink()
    for i, (key, layer_name) in enumerate(layer_map):
        gdf = _sanitize_for_gpkg(export_layers[key])
        mode = 'w' if i == 0 else 'a'
        gdf.to_file(target_path, layer=layer_name, driver="GPKG", mode=mode)


actual_output_gpkg = output_gpkg
try:
    _write_layers(output_gpkg)
except PermissionError:
    print(f"[warn] Cannot overwrite {output_gpkg}. Saving to fallback file.")
    actual_output_gpkg = output_gpkg_fallback
    _write_layers(actual_output_gpkg)

# Keep combined dataframe available for summary and quick checks
output_gdf = beginning.copy()

print(f"\n✓ beginning.gpkg export completed")
print(f"  Output file: {actual_output_gpkg}")
for key, layer_name in layer_map:
    gdf = export_layers[key]
    print(f"  - Layer '{layer_name}': {len(gdf)} records, {len(gdf.columns)} fields")
print(f"  Combined records (all layers): {len(output_gdf)}")


[warn] Cannot overwrite dataset_first_step\beginning.gpkg. Saving to fallback file.

✓ beginning.gpkg export completed
  Output file: dataset_first_step\beginning_border_points.gpkg
  - Layer 'refineries': 4 records, 13 fields
  - Layer 'ports': 11 records, 46 fields
  - Layer 'gas_plants': 5 records, 12 fields
  - Layer 'primary_storage': 19 records, 18 fields
  - Layer 'border_points': 6 records, 29 fields
  Combined records (all layers): 26


# print summary

In [76]:
# STEP 6: Summary
print("\n" + "="*70)
print("PROCESSING COMPLETE")
print("="*70)

print(f"\nConstants: STS_cost={STS_cost}, BUFFER_KM={BUFFER_KM}")
print("LPG_price source: sampled from fob_per_kg.tif at each facility location")
print(f"\nSources by category:")
for cat, count in output_gdf.groupby('category').size().items():
    share = output_gdf[output_gdf['category'] == cat]['percentage'].sum()
    print(f"  - {cat}: {count} sources ({share:.1%} share)")

if 'country' in output_gdf.columns:
    print(f"\nSources by country:")
    for country in sorted(output_gdf['country'].dropna().unique()):
        records = output_gdf[output_gdf['country'] == country]
        print(f"  - {country}: {len(records)} sources ({records['percentage'].sum():.1%})")

print(f"\nOutputs: {data_dir}/")
if (data_dir / 'beginning.gpkg').exists():
    print("  ✓ beginning.gpkg")
if (data_dir / 'beginning_border_points.gpkg').exists():
    print("  ✓ beginning_border_points.gpkg")



PROCESSING COMPLETE

Constants: STS_cost=0.1, BUFFER_KM=10
LPG_price source: sampled from fob_per_kg.tif at each facility location

Sources by category:
  - border_points: 6 sources (0.0% share)
  - gasplants: 5 sources (49.2% share)
  - ports: 11 sources (50.8% share)
  - refineries: 4 sources (0.0% share)

Sources by country:
  - NGA: 20 sources (100.0%)
  - Unknown: 6 sources (0.0%)

Outputs: dataset_first_step/
  ✓ beginning.gpkg
  ✓ beginning_border_points.gpkg
